# Test: does the analysis reproduce from the raw profiles?

Checks one position — `SLG1171_DASTool_bins_9.fa_k141_216652:14297`, allele **T** —
because it is the one site where the answer turns on an absence (zero T reads in
all 15 HF PRE mice), so it is where a bug would matter most.

Independent on purpose: `pre_allele_presence.ipynb` reads the profiles with
`pd.read_csv` using `usecols` / `dtype` / `chunksize`, then merges against a roster
and zero-fills. This re-derives the same four numbers by plain `gzip` line parsing
with no pandas at all, so a mistake in that path cannot hide in this one too.

Passes if the totals match section 4 exactly.

In [1]:
import csv, gzip
from pathlib import Path

RUN = Path("/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_mapq20/longitudinal")
MAG = "SLG1171_DASTool_bins_9"
CONTIG, POSITION, ALLELE = f"{MAG}.fa_k141_216652", 14297, "T"

# From section 4 of pre_allele_presence.ipynb: (allele reads, total depth, mice carrying it)
EXPECTED = {("fat", "pre"):     (0, 159, 0),
            ("fat", "end"):     (10, 145, 6),
            ("control", "pre"): (3, 238, 2),
            ("control", "end"): (4, 151, 3)}

totals, missing = {}, []
meta = RUN / "inputMetadata" / "inputMetadata_pre_end-fat_control" / f"{MAG}_metadata.tsv"

for s in csv.DictReader(open(meta), delimiter="\t"):
    reads = depth = 0
    found = False
    with gzip.open(s["file_path"], "rt") as fh:
        cols = fh.readline().rstrip("\n").split("\t")
        # startswith on "contig\tposition\t" is only valid if those are columns 0 and 1
        assert cols[0] == "contig" and cols[1] == "position", cols[:2]
        i_depth, i_allele = cols.index("total_coverage"), cols.index(ALLELE)
        prefix = f"{CONTIG}\t{POSITION}\t"          # exact match, and fast: no split per line
        for line in fh:
            if line.startswith(prefix):
                f = line.rstrip("\n").split("\t")
                reads, depth, found = int(f[i_allele]), int(f[i_depth]), True
                break
    if not found:
        missing.append(s["sample_id"])              # no row = no high-quality coverage
    r, d, m = totals.get((s["group"], s["time"]), (0, 0, 0))
    totals[(s["group"], s["time"])] = (r + reads, d + depth, m + (reads > 0))

print(f"{CONTIG}:{POSITION}   allele {ALLELE}\n")
print(f"  {'group':8} {'time':5} {'expected':>18} {'got':>18}")
ok = True
for key, exp in EXPECTED.items():
    got = totals[key]
    ok &= got == exp
    print(f"  {key[0]:8} {key[1]:5} {str(exp):>18} {str(got):>18}   {'PASS' if got == exp else 'FAIL'}")

print(f"\nsamples with no row at this position: {missing or 'none'}")
assert ok, "independent scan disagrees with the analysis"
print("\nPASS - section 4 reproduces from the raw profiles.")

SLG1171_DASTool_bins_9.fa_k141_216652:14297   allele T

  group    time            expected                got
  fat      pre          (0, 159, 0)        (0, 159, 0)   PASS
  fat      end         (10, 145, 6)       (10, 145, 6)   PASS
  control  pre          (3, 238, 2)        (3, 238, 2)   PASS
  control  end          (4, 151, 3)        (4, 151, 3)   PASS

samples with no row at this position: ['SLG1155_555']

PASS - section 4 reproduces from the raw profiles.
